# Braket Direct — n-target QPUF analysis

Reads the jobs in `job_results/job_log.txt` and the retrieved counts in
`job_results/<uuid>.json` (produced by `checkRetrieve.py`), then analyses
the two-stage QPE QPUF results for `n_targ >= 1`.

Modeled on `garnet-approxQPEQPUF-1target/result_analysis.ipynb`, generalised
to handle multi-target Haar-random unitaries: each `U` has `2^n_targ`
eigenphases, so the ideal QPE peaks form a set rather than a single bin.

In [ ]:
import json
import os
import glob

import numpy as np
import matplotlib.pyplot as plt

JOB_RESULTS_DIR = os.path.join(os.getcwd(), 'job_results')
LOG_FILE        = os.path.join(JOB_RESULTS_DIR, 'job_log.txt')

print('job_results dir :', JOB_RESULTS_DIR)
print('job_log.txt     :', LOG_FILE, ' (exists:', os.path.exists(LOG_FILE), ')')

## Analysis helpers

In [ ]:
def decode_unitary(rec: dict) -> np.ndarray:
    """Reconstruct the complex Haar U from the serialized real/imag lists."""
    u = rec['unitary']
    return np.array(u['real']) + 1j * np.array(u['imag'])


def parse_outcome(bitstring: str, n_prec: int) -> tuple[int, int]:
    """Counts key -> (m1, m2). Handles 'c2 c1' or concatenated layout."""
    parts = bitstring.split(' ')
    if len(parts) == 2:
        return int(parts[1], 2), int(parts[0], 2)
    return int(bitstring[n_prec:], 2), int(bitstring[:n_prec], 2)


def cyclic_distance(a: int, b: int, n_prec: int) -> int:
    M    = 2 ** n_prec
    diff = abs(a - b)
    return min(diff, M - diff)


def eigenphase_bins(U: np.ndarray, n_prec: int) -> list[int]:
    """Sorted list of distinct ideal QPE bins from eig(U)."""
    eigvals = np.linalg.eig(U)[0]
    phases  = np.mod(np.angle(eigvals) / (2 * np.pi), 1.0)
    return sorted({round(float(p) * 2 ** n_prec) % 2 ** n_prec for p in phases})


def nearest_eigenbin_distance(m: int, bins: list[int], n_prec: int) -> int:
    """Cyclic distance from m to the nearest ideal eigenbin."""
    return min(cyclic_distance(m, b, n_prec) for b in bins)


def analyse_counts(counts: dict, n_prec: int, U: np.ndarray, delta: int = 2,
                   label: str = '') -> dict:
    """Print acceptance stats and top outcomes. Returns a stats dict."""
    bins  = eigenphase_bins(U, n_prec)
    total = sum(counts.values())

    consistent = 0   # m1 ≈ m2
    near_eig   = 0   # both m1 and m2 near some eigenbin
    for outcome, cnt in counts.items():
        m1, m2 = parse_outcome(outcome, n_prec)
        if cyclic_distance(m1, m2, n_prec) <= delta:
            consistent += cnt
        if (nearest_eigenbin_distance(m1, bins, n_prec) <= delta and
            nearest_eigenbin_distance(m2, bins, n_prec) <= delta):
            near_eig += cnt

    prefix = f'[{label}] ' if label else ''
    print(f'{prefix}Total shots                : {total}')
    print(f'{prefix}|m1 - m2|_cyclic <= {delta}      : {consistent} ({consistent/total:.4f})')
    print(f'{prefix}both near eigenbin (<= {delta}) : {near_eig} ({near_eig/total:.4f})')
    print(f'{prefix}Ideal eigenbins             : {bins}')

    print(f'\n{prefix}Top 10 outcomes:')
    print(f'  {"bitstring":28s}  m1   m2   dist  count')
    print(f'  {"-"*54}')
    for k, v in sorted(counts.items(), key=lambda x: -x[1])[:10]:
        m1, m2 = parse_outcome(k, n_prec)
        print(f'  {k!r:28s}  {m1:4d} {m2:4d}  {cyclic_distance(m1, m2, n_prec):4d}  {v}')

    return {
        'total':       total,
        'consistent':  consistent,
        'near_eig':    near_eig,
        'bins':        bins,
        'delta':       delta,
    }

print('Analysis helpers loaded.')

## Plotting helpers

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

def plot_m1_vs_m2_3d(counts: dict, n_prec: int, U: np.ndarray, n_shots: int,
                      title_suffix: str = '', ax=None):
    """3D bar chart: X=m1, Y=m1-m2 (signed), Z=frequency (%)."""
    total = sum(counts.values())

    # Aggregate into (m1, m1-m2) buckets
    data = {}
    for outcome, cnt in counts.items():
        m1, m2 = parse_outcome(outcome, n_prec)
        diff = m1 - m2
        key = (m1, diff)
        data[key] = data.get(key, 0) + cnt

    bins = eigenphase_bins(U, n_prec)
    M    = 2 ** n_prec

    if ax is None:
        fig = plt.figure(figsize=(8, 6))
        ax  = fig.add_subplot(111, projection='3d')

    xs = np.array([k[0] for k in data])
    ys = np.array([k[1] for k in data])
    zs = np.array([data[k] / total * 100 for k in data])

    dx = max(M / 80, 1)
    dy = max(M / 80, 1)

    ax.bar3d(xs, ys, np.zeros_like(zs), dx, dy, zs,
             shade=True, alpha=0.75, color='steelblue', zsort='average')

    for b in bins:
        y_range = M - 1
        ax.plot([b, b], [-y_range, y_range], [0, 0], color='green', linestyle=':', alpha=0.5)

    ax.set_xlabel('m1  (Stage 1 QPE)')
    ax.set_ylabel('m1 − m2')
    ax.set_zlabel('Frequency of outcome (%)')
    title = f'm1 vs m1−m2  (n_prec={n_prec}, shots={n_shots})'
    if title_suffix:
        title += f'  —  {title_suffix}'
    ax.set_title(title)
    return ax


def plot_acceptance_vs_delta(counts: dict, n_prec: int, n_shots: int,
                              delta_mark: int = 2, ax=None,
                              title_suffix: str = ''):
    """Acceptance rate (|m1-m2|_cyclic <= delta) vs delta."""
    total       = sum(counts.values())
    delta_range = list(range(0, 2 ** n_prec + 1))
    acc_rates   = [
        sum(cnt for outcome, cnt in counts.items()
            if cyclic_distance(*parse_outcome(outcome, n_prec), n_prec) <= d) / total
        for d in delta_range
    ]
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(delta_range, acc_rates, marker='o', markersize=3)
    ax.axvline(delta_mark, color='r', linestyle='--', label=f'delta={delta_mark}')
    ax.set_xlabel('delta  (cyclic distance threshold)')
    ax.set_ylabel('acceptance rate')
    title = f'Acceptance vs delta (n_prec={n_prec}, shots={n_shots})'
    if title_suffix:
        title += f'  -  {title_suffix}'
    ax.set_title(title)
    ax.legend()
    return ax


def plot_hw_vs_sim(hw: dict, sim: dict, U: np.ndarray):
    """2x2 comparison: 3D bar chart and delta-curve, hardware vs simulator."""
    fig = plt.figure(figsize=(13, 11))
    ax00 = fig.add_subplot(2, 2, 1, projection='3d')
    ax01 = fig.add_subplot(2, 2, 2, projection='3d')
    ax10 = fig.add_subplot(2, 2, 3)
    ax11 = fig.add_subplot(2, 2, 4)

    plot_m1_vs_m2_3d(hw['counts'],  hw['n_prec'],  U, hw['n_shots'],
                      title_suffix=hw.get('label', 'Hardware'),    ax=ax00)
    plot_m1_vs_m2_3d(sim['counts'], sim['n_prec'], U, sim['n_shots'],
                      title_suffix=sim.get('label', 'Simulator'),  ax=ax01)
    plot_acceptance_vs_delta(hw['counts'],  hw['n_prec'],  hw['n_shots'],
                              delta_mark=hw.get('delta', 2),
                              title_suffix=hw.get('label', 'Hardware'),  ax=ax10)
    plot_acceptance_vs_delta(sim['counts'], sim['n_prec'], sim['n_shots'],
                              delta_mark=sim.get('delta', 2),
                              title_suffix=sim.get('label', 'Simulator'), ax=ax11)
    plt.tight_layout()
    return fig

print('Plot helpers loaded.')

## Local-simulator helper (re-runs the same `U` on AerSimulator)

Reuses the exact circuit builders from `submit_QPUF_ntarg.py` so the
hardware and simulator runs are byte-identical at the circuit level.

In [ ]:
from submit_QPUF_ntarg import build_full_circuit
from qiskit import transpile

def run_local_sim(rec: dict, U: np.ndarray, n_shots: int | None = None,
                  delta: int = 2) -> dict:
    """Run the same circuit (same U, same target-init seed) on AerSimulator."""
    try:
        from qiskit_aer import AerSimulator
    except ImportError as e:
        raise RuntimeError('qiskit-aer not installed') from e

    n_prec   = rec['n_prec']
    n_targ   = rec['n_targ']
    init_sd  = rec.get('target_init_seed', 99)
    shots    = n_shots if n_shots is not None else rec['n_shots']

    qc = build_full_circuit(n_prec, n_targ, U, target_init_seed=init_sd)
    sim = AerSimulator()
    qc_sim = transpile(qc, sim)
    counts = sim.run(qc_sim, shots=shots).result().get_counts()

    return {
        'counts':  counts,
        'n_prec':  n_prec,
        'n_targ':  n_targ,
        'n_shots': shots,
        'delta':   delta,
        'label':   'Local Simulator',
    }

print('Local-sim helper loaded.')

## Noisy-simulator helper (depolarizing noise on rz/rx/rxx)

Same `U` and same target-init seed as `run_local_sim`, but with depolarizing
errors attached to the IonQ-like native basis (`rz`, `rx`, `rxx`). The
fidelities are surfaced as module-level constants so per-qubit / readout /
dephasing noise can be added later without rewriting the helper.

Average-gate-fidelity → depolarizing parameter (qiskit-aer convention):

    F_avg = 1 - p * (d - 1) / d
    1q (d=2): p = 2 (1 - F)
    2q (d=4): p = 4 (1 - F) / 3


In [ ]:
# Fidelity knobs — swap these later for per-qubit / readout / dephasing models.
ONE_Q_FIDELITY = 0.9999     # ~IonQ Forte-1 single-qubit native
TWO_Q_FIDELITY = 0.9925     # ~IonQ Forte-1 two-qubit native

NOISY_BASIS = ['rz', 'rx', 'rxx', 'measure', 'reset']


def _depol_param(fidelity: float, n_qubits: int) -> float:
    """Convert average gate fidelity to qiskit-aer depolarizing param."""
    d = 2 ** n_qubits
    return (1 - fidelity) * d / (d - 1)


def build_ionq_noise_model(one_q_fidelity: float = ONE_Q_FIDELITY,
                            two_q_fidelity: float = TWO_Q_FIDELITY):
    """Depolarizing noise on the IonQ-like native basis."""
    from qiskit_aer.noise import NoiseModel, depolarizing_error

    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(
        depolarizing_error(_depol_param(one_q_fidelity, 1), 1),
        ['rz', 'rx'],
    )
    nm.add_all_qubit_quantum_error(
        depolarizing_error(_depol_param(two_q_fidelity, 2), 2),
        ['rxx'],
    )
    return nm


def run_local_sim_noisy(rec: dict, U: np.ndarray, n_shots: int | None = None,
                        delta: int = 2,
                        one_q_fidelity: float = ONE_Q_FIDELITY,
                        two_q_fidelity: float = TWO_Q_FIDELITY,
                        n_prec_override: int | None = None) -> dict:
    """Same circuit as run_local_sim, but transpiled to the IonQ basis and run
    on AerSimulator with a depolarizing noise model. n_prec_override lets the
    shot-sweep study use a smaller register (statevector sim of 33 qubits at
    N_PREC=30 is intractable)."""
    try:
        from qiskit_aer import AerSimulator
    except ImportError as e:
        raise RuntimeError('qiskit-aer not installed') from e

    n_prec   = n_prec_override if n_prec_override is not None else rec['n_prec']
    n_targ   = rec['n_targ']
    init_sd  = rec.get('target_init_seed', 99)
    shots    = n_shots if n_shots is not None else rec['n_shots']

    qc     = build_full_circuit(n_prec, n_targ, U, target_init_seed=init_sd)
    qc_hw  = transpile(qc, basis_gates=NOISY_BASIS, optimization_level=1)
    noise  = build_ionq_noise_model(one_q_fidelity, two_q_fidelity)
    sim    = AerSimulator(noise_model=noise)
    counts = sim.run(qc_hw, shots=shots).result().get_counts()

    return {
        'counts':  counts,
        'n_prec':  n_prec,
        'n_targ':  n_targ,
        'n_shots': shots,
        'delta':   delta,
        'label':   f'Noisy Sim (F1q={one_q_fidelity}, F2q={two_q_fidelity})',
        'one_q_fidelity': one_q_fidelity,
        'two_q_fidelity': two_q_fidelity,
    }


print(f'Noisy-sim helper loaded.  1q depol param = {_depol_param(ONE_Q_FIDELITY, 1):.2e}, '
      f'2q depol param = {_depol_param(TWO_Q_FIDELITY, 2):.2e}')


## List submitted jobs (from `job_log.txt`)

In [ ]:
with open(LOG_FILE) as f:
    submitted = [json.loads(line) for line in f if line.strip()]

print(f'{len(submitted)} job(s) recorded in job_log.txt:\n')
print(f"  {'idx':>3}  {'datetime':25s}  {'qpu':20s}  {'type':18s}  n_prec  n_targ  n_shots  uuid")
print('  ' + '-' * 110)
for i, r in enumerate(submitted):
    uuid = r['job_id'].split('/')[-1]
    print(f"  {i:>3}  {r['datetime'][:25]:25s}  {r['qpu']:20s}  "
          f"{r.get('circuit_type','-'):18s}  {r['n_prec']:>6}  {r.get('n_targ','-'):>6}  "
          f"{r['n_shots']:>7}  {uuid[:8]}...")

## List retrieved results (`job_results/*.json` written by `checkRetrieve.py`)

In [ ]:
retrieved_paths = sorted(glob.glob(os.path.join(JOB_RESULTS_DIR, '*.json')))
retrieved = []
for p in retrieved_paths:
    with open(p) as f:
        retrieved.append(json.load(f))

print(f'{len(retrieved)} retrieved result(s):\n')
for i, r in enumerate(retrieved):
    uuid = r['job_id'].split('/')[-1]
    print(f"  [{i}]  {uuid[:8]}...  {r.get('circuit_type','-')}  "
          f"n_prec={r['n_prec']}  n_targ={r.get('n_targ','-')}  "
          f"shots={r['n_shots']}  unique_outcomes={len(r['counts'])}")

if not retrieved:
    print('(none yet — run `python checkRetrieve.py` after the job completes)')

## Pick one retrieved job and analyse

In [ ]:
# Choose which retrieved job to analyse. Default: most recent.
JOB_IDX = 3
DELTA   = 2

assert retrieved, 'No retrieved results yet. Run checkRetrieve.py first.'
rec     = retrieved[JOB_IDX]
U       = decode_unitary(rec)
counts  = rec['counts']
n_prec  = rec['n_prec']
n_targ  = rec['n_targ']
n_shots = rec['n_shots']
uuid    = rec['job_id'].split('/')[-1]

print(f'Job UUID    : {uuid}')
print(f'Submitted   : {rec["datetime"]}')
print(f'QPU         : {rec["qpu"]}')
print(f'Type        : {rec.get("circuit_type","-")}')
print(f'n_prec      : {n_prec}')
print(f'n_targ      : {n_targ}  (U is {2**n_targ}×{2**n_targ})')
print(f'shots       : {n_shots}')
print(f'transp gates: {rec.get("n_gates","-")}')
print(f'per_U_gates : {rec.get("per_U_gates","-")}  (transpiled gates per ctrl-U)')
u_err = float(np.max(np.abs(U.conj().T @ U - np.eye(U.shape[0]))))
print(f'|U†U − I|   : {u_err:.2e}  (sanity: stored U is unitary)')

In [ ]:
stats_hw = analyse_counts(counts, n_prec, U, delta=DELTA, label='hw')

In [ ]:
PLOTS_DIR = os.path.join(os.getcwd(), 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

fig = plt.figure(figsize=(13, 5.5))
ax0 = fig.add_subplot(1, 2, 1, projection='3d')
ax1 = fig.add_subplot(1, 2, 2)
plot_m1_vs_m2_3d(counts, n_prec, U, n_shots, title_suffix='Hardware', ax=ax0)
plot_acceptance_vs_delta(counts, n_prec, n_shots, delta_mark=DELTA,
                          title_suffix='Hardware', ax=ax1)
plt.tight_layout()
fname = os.path.join(PLOTS_DIR, f'hw_{uuid[:8]}_nprec{n_prec}_ntarg{n_targ}.png')
fig.savefig(fname, dpi=150, bbox_inches='tight')
print(f'Saved: {fname}')
plt.show()

## Hardware vs Local Simulator

Rebuilds the same circuit with the stored `U` and runs it on AerSimulator,
so any divergence between the two panels is attributable to device noise
(not to a circuit-construction bug).

In [ ]:
sim = run_local_sim(rec, U, n_shots=max(n_shots, 1000), delta=DELTA)
stats_sim = analyse_counts(sim['counts'], n_prec, U, delta=DELTA, label='sim')

hw = {
    'counts':  counts, 'n_prec': n_prec, 'n_targ': n_targ,
    'n_shots': n_shots, 'delta': DELTA, 'label': f'Hardware ({rec["qpu"]})',
}
fig = plot_hw_vs_sim(hw, sim, U)
fname = os.path.join(PLOTS_DIR, f'hw_vs_sim_{uuid[:8]}_nprec{n_prec}_ntarg{n_targ}.png')
fig.savefig(fname, dpi=150, bbox_inches='tight')
print(f'Saved: {fname}')
plt.show()

## Shot-sweep uniformity study

Goal: find the shot count beyond which more shots adds no information,
because the noisy distribution has converged to its (near-uniform) shape.

Three metrics:
  - **Normalized entropy**  `H(p)/log(M)` — 1.0 = uniform, 0 = single peak.
    M = 2^(2·n_prec) is the full outcome space. At low shots the empirical
    entropy is bounded above by `log(n_shots)/log(M)`, plotted as a dashed
    reference line. Useful comparison is noisy vs noiseless on the same axes.
  - **KL-to-uniform**  `KL(p ‖ uniform)` — drops toward 0 as p flattens.
  - **TV(noisy, noiseless)** — intrinsic noise gap; once both sims are
    well-sampled this plateaus. Plateau onset = "more shots no longer
    changes the comparison."


In [ ]:
def counts_to_probs(counts: dict) -> dict:
    total = sum(counts.values())
    return {k: v / total for k, v in counts.items()}


def tv_distance(counts_a: dict, counts_b: dict) -> float:
    """Total variation distance ½ Σ |p_a − p_b| over the union of supports."""
    pa, pb = counts_to_probs(counts_a), counts_to_probs(counts_b)
    keys   = set(pa) | set(pb)
    return 0.5 * sum(abs(pa.get(k, 0.0) - pb.get(k, 0.0)) for k in keys)


def normalized_entropy(counts: dict, n_prec: int) -> float:
    """H(p)/log(M), M = 2^(2·n_prec). 0 ≤ value ≤ min(1, log(n_shots)/log(M))."""
    total  = sum(counts.values())
    ps     = np.array([c / total for c in counts.values()])
    H      = -np.sum(ps * np.log(ps))
    M      = 2.0 ** (2 * n_prec)
    return float(H / np.log(M))


def kl_to_uniform(counts: dict, n_prec: int) -> float:
    """KL(p ‖ uniform). Uniform has prob 1/M on each of M outcomes."""
    total  = sum(counts.values())
    M      = 2.0 ** (2 * n_prec)
    ps     = np.array([c / total for c in counts.values()])
    return float(np.sum(ps * (np.log(ps) + np.log(M))))


print('Sweep metrics loaded.')


### Run the sweep

`SWEEP_N_PREC` is intentionally small (default 6) — AerSimulator with noise
on a 30-prec / 3-targ circuit (33 qubits) is intractable. The qualitative
behaviour (where the noisy curve flattens relative to the noiseless one)
transfers to the full-size hardware run.

Each shot count runs both noiseless and noisy sims once. Bump `SWEEP_TRIALS`
for averaging.


In [ ]:
SWEEP_N_PREC  = 5                                      # override for tractability
SHOT_GRID     = [100, 300, 1000, 3000, 10000, 20000]
SWEEP_TRIALS  = 1                                       # repeats per shot count (averaged)

# Reuse U + init-seed from the loaded record if present; otherwise sample one.
if 'rec' in globals():
    U_sweep        = U
    init_seed_swp  = rec.get('target_init_seed', 99)
    n_targ_swp     = rec['n_targ']
else:
    from submit_QPUF_ntarg import haar_random_unitary
    rng_swp        = np.random.default_rng(seed=100)
    n_targ_swp     = 3
    U_sweep        = haar_random_unitary(2 ** n_targ_swp, rng=rng_swp)
    init_seed_swp  = 99

rec_swp = {'n_prec': SWEEP_N_PREC, 'n_targ': n_targ_swp,
            'target_init_seed': init_seed_swp, 'n_shots': max(SHOT_GRID)}

sweep_rows = []
for shots in SHOT_GRID:
    for trial in range(SWEEP_TRIALS):
        noiseless = run_local_sim       (rec_swp, U_sweep, n_shots=shots)
        noisy     = run_local_sim_noisy (rec_swp, U_sweep, n_shots=shots,
                                          n_prec_override=SWEEP_N_PREC)
        sweep_rows.append({
            'shots':         shots,
            'trial':         trial,
            'H_noiseless':   normalized_entropy(noiseless['counts'], SWEEP_N_PREC),
            'H_noisy':       normalized_entropy(noisy['counts'],     SWEEP_N_PREC),
            'KL_noiseless':  kl_to_uniform     (noiseless['counts'], SWEEP_N_PREC),
            'KL_noisy':      kl_to_uniform     (noisy['counts'],     SWEEP_N_PREC),
            'TV_noisy_vs_nl': tv_distance(noisy['counts'], noiseless['counts']),
            'acc_noiseless': sum(c for o, c in noiseless['counts'].items()
                                 if cyclic_distance(*parse_outcome(o, SWEEP_N_PREC),
                                                     SWEEP_N_PREC) <= DELTA)
                              / sum(noiseless['counts'].values()),
            'acc_noisy':     sum(c for o, c in noisy['counts'].items()
                                 if cyclic_distance(*parse_outcome(o, SWEEP_N_PREC),
                                                     SWEEP_N_PREC) <= DELTA)
                              / sum(noisy['counts'].values()),
        })
    print(f'  shots={shots:>7}  done')

# Average across trials (per-shot-count means).
def _mean(field):
    out = {}
    for r in sweep_rows:
        out.setdefault(r['shots'], []).append(r[field])
    return {s: float(np.mean(v)) for s, v in out.items()}

run = False

if run is True:
    mH_nl   = _mean('H_noiseless'  ); mH_n   = _mean('H_noisy')mKL_nl  = _mean('KL_noiseless' )
    mKL_n  = _mean('KL_noisy')mTV     = _mean('TV_noisy_vs_nl')mAcc_nl = _mean('acc_noiseless')
    mAcc_n = _mean('acc_noisy')
    print('\nSweep complete.')

else:
    print("Skipped")

In [ ]:
shots_sorted = sorted(mH_nl)
M_full       = 2.0 ** (2 * SWEEP_N_PREC)
H_ceiling    = [np.log(s) / np.log(M_full) for s in shots_sorted]

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(shots_sorted, [mH_nl[s] for s in shots_sorted], 'o-', label='noiseless')
ax.plot(shots_sorted, [mH_n [s] for s in shots_sorted], 's-', label='noisy')
ax.plot(shots_sorted, H_ceiling, 'k--', alpha=0.4,
         label='log(shots)/log(M)  (sampling ceiling)')
ax.set_xscale('log'); ax.set_xlabel('shots'); ax.set_ylabel('H(p)/log(M)')
ax.set_title(f'Normalized entropy vs shots  (n_prec={SWEEP_N_PREC})')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()